# B2B Radar — read-only result analysis

This notebook reads checksum-verified artifacts from one completed or awaiting-review ML run. It never invokes the pipeline and never writes inside `ml-runs`. Aggregate reports are optional; private text is hidden by default.

In [ ]:
REPOSITORY_URL = "https://github.com/osmirnov34/b2b-radar.git"
CODE_REF = "main"  # Prefer the commit recorded by the source run.
PROJECT_DIR = "/content/b2b-radar-reporting"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/b2b-radar"
RUN_ID = "colab-full-001"
MAX_SCATTER_POINTS = 20_000
TOP_CLUSTERS = 30
SAVE_REPORT = False
OVERWRITE_REPORT = False
SHOW_PRIVATE_TEXT = False

In [ ]:
import json
import platform
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13)}:
    raise RuntimeError(f"Unsupported Python {platform.python_version()}; expected 3.11-3.13")
project_root = Path(PROJECT_DIR)
if project_root.exists():
    raise FileExistsError(f"Reporting checkout already exists; restart the runtime: {project_root}")
subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(project_root)], check=True)
subprocess.run(["git", "-C", str(project_root), "checkout", CODE_REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{project_root}[visualization]"], check=True)
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from google.colab import drive

from src.ml.reporting import (
    load_analysis_artifacts,
    processing_flow,
    stratified_plot_indices,
    topic_summary_rows,
    write_analysis_tables,
)

drive.mount("/content/drive")
run_dir = Path(DRIVE_PROJECT_DIR) / "ml-runs" / RUN_ID
artifacts = load_analysis_artifacts(run_dir)
report_root = Path(DRIVE_PROJECT_DIR) / "visualizations"
report_manifest = (
    write_analysis_tables(artifacts, report_root, overwrite=OVERWRITE_REPORT) if SAVE_REPORT else None
)
report_dir = report_root / RUN_ID
figures_dir = report_dir / "figures"
if SAVE_REPORT:
    figures_dir.mkdir(parents=True, exist_ok=True)
print(artifacts.summary.model_dump())

In [ ]:
flow_frame = pd.DataFrame([step.model_dump() for step in processing_flow(artifacts)])
display(flow_frame)
fig_flow = px.bar(
    flow_frame,
    x="stage",
    y="records",
    text="records",
    hover_data=["note"],
    title="Record flow through the pipeline",
)
fig_flow.show()
if SAVE_REPORT:
    fig_flow.write_html(figures_dir / "processing-flow.html", include_plotlyjs="cdn")

In [ ]:
topic_rows = topic_summary_rows(artifacts)
topic_frame = pd.DataFrame([row.model_dump() for row in topic_rows])
topic_frame["keywords"] = topic_frame["keywords"].map(" | ".join)
display(topic_frame.sort_values("records", ascending=False).head(TOP_CLUSTERS))

In [ ]:
labels = np.asarray(artifacts.labels)
confidence = np.asarray(artifacts.confidence)
cluster_counts = Counter(int(label) for label in labels if label >= 0)
largest = cluster_counts.most_common(TOP_CLUSTERS)
names = {row.topic_id: row.name for row in topic_rows}
fig_overview, axes = plt.subplots(1, 3, figsize=(22, 7))
axes[0].barh(range(len(largest)), [count for _, count in largest], color="steelblue")
axes[0].set_yticks(
    range(len(largest)),
    [f"{topic}: {names.get(topic, '')[:28]}" for topic, _ in largest],
)
axes[0].invert_yaxis()
axes[0].set_title("Largest topics")
axes[0].set_xlabel("Records")
axes[1].pie(
    [len(labels) - artifacts.summary.outliers, artifacts.summary.outliers],
    labels=["Clustered", "Outliers"],
    autopct="%1.1f%%",
    colors=["steelblue", "lightgray"],
)
axes[1].set_title("Outlier share")
axes[2].hist(confidence, bins=30, color="seagreen", alpha=0.8)
axes[2].set_title("Assignment confidence")
axes[2].set_xlabel("Confidence")
plt.tight_layout()
if SAVE_REPORT:
    fig_overview.savefig(figures_dir / "cluster-overview.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
plot_indices = stratified_plot_indices(labels, maximum=min(MAX_SCATTER_POINTS, len(labels)), seed=42)
coordinates = np.asarray(artifacts.coordinates[plot_indices, :2])
plot_labels = labels[plot_indices]
plot_frame = pd.DataFrame({
    "x": coordinates[:, 0], "y": coordinates[:, 1],
    "topic_id": [str(int(value)) for value in plot_labels],
    "topic": ["Outlier" if value == -1 else names.get(int(value), "unnamed") for value in plot_labels],
    "confidence": confidence[plot_indices],
})
fig_umap = px.scatter(
    plot_frame,
    x="x",
    y="y",
    color="topic_id",
    hover_data=["topic", "confidence"],
    opacity=0.65,
    render_mode="webgl",
    title="Stratified UMAP cluster view",
)
fig_umap.show()
if SAVE_REPORT:
    fig_umap.write_html(figures_dir / "umap-clusters.html", include_plotlyjs="cdn")

In [ ]:
shown_topics = sorted(artifacts.topics, key=lambda topic: topic.records, reverse=True)[:TOP_CLUSTERS]
vocabulary = sorted({keyword.term for topic in shown_topics for keyword in topic.keywords})
term_index = {term: index for index, term in enumerate(vocabulary)}
topic_vectors = np.zeros((len(shown_topics), len(vocabulary)), dtype=np.float32)
for row, topic in enumerate(shown_topics):
    for keyword in topic.keywords:
        topic_vectors[row, term_index[keyword.term]] = keyword.weight
norms = np.linalg.norm(topic_vectors, axis=1, keepdims=True)
normalized = np.divide(topic_vectors, norms, out=np.zeros_like(topic_vectors), where=norms != 0)
similarity = normalized @ normalized.T
topic_labels = [f"{topic.topic_id}: {topic.name[:24]}" for topic in shown_topics]
fig_similarity = px.imshow(
    similarity,
    x=topic_labels,
    y=topic_labels,
    zmin=0,
    zmax=1,
    color_continuous_scale="Blues",
    title="Topic keyword similarity",
)
fig_similarity.show()
if SAVE_REPORT:
    fig_similarity.write_html(figures_dir / "topic-similarity.html", include_plotlyjs="cdn")

In [ ]:
topic_languages = defaultdict(Counter)
topic_queries = defaultdict(Counter)
topic_videos = defaultdict(Counter)
topic_months = defaultdict(Counter)
representative_text = {}
representative_indexes = {index for topic in artifacts.topics for index in topic.representative_indices}
with artifacts.corpus_path.open(encoding="utf-8") as source:
    record_index = -1
    for line in source:
        if not line.strip():
            continue
        record_index += 1
        record = json.loads(line)
        topic_id = int(labels[record_index])
        topic_languages[topic_id][record.get("detected_language", "unknown")] += 1
        topic_queries[topic_id][record.get("search_query") or "unknown"] += 1
        topic_videos[topic_id][record.get("video_id") or "unknown"] += 1
        published_at = record.get("published_at")
        if published_at:
            topic_months[topic_id][published_at[:7]] += 1
        if SHOW_PRIVATE_TEXT and record_index in representative_indexes:
            representative_text[record_index] = record.get("text", "")
source_rows = []
for topic_id, count in largest:
    source_rows.append({
        "topic_id": topic_id, "topic": names.get(topic_id, ""),
        "top_language": topic_languages[topic_id].most_common(1)[0][0],
        "top_query_share": topic_queries[topic_id].most_common(1)[0][1] / count,
        "top_video_share": topic_videos[topic_id].most_common(1)[0][1] / count,
    })
source_frame = pd.DataFrame(source_rows)
display(source_frame)
language_frame = pd.DataFrame(
    {topic_id: dict(values) for topic_id, values in topic_languages.items()}
).fillna(0)
language_share = language_frame.div(language_frame.sum(axis=0), axis=1)
fig_languages = px.imshow(
    language_share,
    labels={"x": "Topic ID", "y": "Language", "color": "Share"},
    title="Language composition by topic",
)
fig_languages.show()
time_rows = [
    {"topic_id": topic_id, "month": month, "records": records}
    for topic_id, months in topic_months.items()
    for month, records in months.items()
    if topic_id in dict(largest[:10])
]
if time_rows:
    time_frame = pd.DataFrame(time_rows).sort_values("month")
    fig_time = px.line(
        time_frame,
        x="month",
        y="records",
        color="topic_id",
        markers=True,
        title="Largest topic activity by month",
    )
    fig_time.show()
if SHOW_PRIVATE_TEXT:
    for topic in shown_topics:
        print({
            "topic_id": topic.topic_id,
            "name": topic.name,
            "representatives": [
                representative_text.get(index, "")
                for index in topic.representative_indices
            ],
        })
else:
    print("Representative text is hidden. Set SHOW_PRIVATE_TEXT=True only for local manual review.")
if SAVE_REPORT:
    source_frame.to_csv(report_dir / "tables/source-concentration.csv", index=False)
    fig_languages.write_html(figures_dir / "topic-languages.html", include_plotlyjs="cdn")
    if time_rows:
        fig_time.write_html(figures_dir / "topic-timeline.html", include_plotlyjs="cdn")

In [ ]:
review_frame = topic_frame[["topic_id", "name", "records", "keywords"]].copy()
review_frame["clarity_1_to_5"] = ""
review_frame["homogeneity_1_to_5"] = ""
review_frame["decision"] = ""  # keep / rename / merge / split / reject
review_frame["reviewer_name"] = ""
review_frame["notes"] = ""
display(review_frame.head(TOP_CLUSTERS))
if SAVE_REPORT:
    (report_dir / "review").mkdir(parents=True, exist_ok=True)
    review_frame.to_csv(report_dir / "review/manual-topic-review.csv", index=False)

In [ ]:
if SAVE_REPORT:
    manifest_path = report_dir / "report-manifest.json"
    saved_files = sorted(str(path) for path in report_dir.rglob("*") if path.is_file())
    updated_manifest = report_manifest.model_copy(update={"files": saved_files})
    manifest_path.write_text(f"{updated_manifest.model_dump_json(indent=2)}\n", encoding="utf-8")
    print({"report": str(report_dir), "files": len(saved_files)})
else:
    print("Read-only analysis completed; SAVE_REPORT=False, so no report files were written.")